NUESTROS DATOS LOS HEMOS OBTENIDO A TRAVES DEL PORTAL DE ENCUESTAS EUROPEO https://ess.sikt.no/en/data-builder/?tab=variables.
NUESTRA HIPOTESIS NULA ES QUE EL ESTADO ECONOMICO  Y LA CULTURA REGIONAL INFLUYE EN LA OPINION GENERAL Y EN LA ADOPCION DE IDEOLOGIAS POLITICAS.
Tenemos 3 grandes grupos respecto a la Media economica europea, teniendo en cuenta la inflacion y el sueldo bruto anual.
Nuestra Variable target será 'lrscale' escala de ideologia politica 0=extrema izquierda y 10= extrema derecha.
Grupo 1: Alemania (Superior a la Media Europea)
Grupo 2: España (Media Europea)
Grupo 3: Hungria  (Inferior a la media)

In [3]:
import pandas as pd
import random
import numpy as np, random
random.seed(42)

In [4]:
#Cargamos todos los csv.
#Grupo 1:
df_deu= pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/DeuData.csv')

In [5]:
df_deu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36845 entries, 0 to 36844
Data columns (total 27 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      36845 non-null  object 
 1   essround  36845 non-null  int64  
 2   edition   36845 non-null  float64
 3   proddate  36845 non-null  object 
 4   idno      36845 non-null  int64  
 5   cntry     36845 non-null  object 
 6   dweight   36845 non-null  float64
 7   pspwght   36845 non-null  float64
 8   pweight   36845 non-null  float64
 9   anweight  31059 non-null  float64
 10  prob      13503 non-null  float64
 11  stratum   13503 non-null  float64
 12  psu       13503 non-null  float64
 13  pplhlp    36845 non-null  int64  
 14  ppltrst   36845 non-null  int64  
 15  lawobey   2919 non-null   float64
 16  lrscale   36845 non-null  int64  
 17  stfgov    36845 non-null  int64  
 18  trstep    36845 non-null  int64  
 19  trstlgl   36845 non-null  int64  
 20  trstplc   36845 non-null  in

In [6]:
#Eliminamos columnas del df que no interesen
columnas_a_eliminar = ['name', 'essround', 'edition', 'proddate', 'idno', 'dweight', 'pspwght', 'pweight', 'anweight', 'prob', 'stratum', 'psu']
df_deu.drop(columnas_a_eliminar, axis=1, inplace=True)

In [7]:
#Obtenemos una lista con los nombres de las columnas restantes.
nombres_columnas = df_deu.columns.tolist()
print(nombres_columnas)

['cntry', 'pplhlp', 'ppltrst', 'lawobey', 'lrscale', 'stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstun', 'trstsci', 'happy']


In [8]:

#Observamos que las variables lawobey y trstsci son las que tienen datos faltantes.
import pandas as pd
import numpy as np

datadeu = {
    'lawobey': {
        1: 0.019,
        2: 0.532,
        3: 0.149,
        4: 0.115,
        5: 0.012,
    },
    'trstsci': {
        0: 0.006,
        1: 0.002,
        2: 0.011,
        3: 0.012,
        4: 0.019,
        5: 0.062,
        6: 0.09,
        7: 0.246,
        8: 0.324,
        9: 0.185,
        10: 0.046,
    }
}

def rngpond(df, datadeu):
    """
    Rellena los datos faltantes en un DataFrame usando valores aleatorios
    generados según los pesos ponderados proporcionados en el diccionario datadeu.
    """
    for columna, pesos in datadeu.items():
        if columna in df.columns and df[columna].isnull().any():
            valores = np.array(list(pesos.keys()))
            pesos_ponderados = np.array(list(pesos.values()))

            # Normalizar los pesos ponderados
            suma_pesos = np.sum(pesos_ponderados)
            pesos_normalizados = pesos_ponderados / suma_pesos

            for index, row in df[df[columna].isnull()].iterrows():
                # Generar un valor aleatorio según los pesos ponderados normalizados
                valor_aleatorio = np.random.choice(valores, p=pesos_normalizados)

                # Rellenar el valor faltante con el valor aleatorio generado
                df.loc[index, columna] = valor_aleatorio
    return df

# Supongamos que df_deu es tu DataFrame original
# df_deu = ... # Carga tu DataFrame aquí

df_deuv2 = rngpond(df_deu.copy(), datadeu)

print(df_deuv2.isnull().sum())

cntry         0
pplhlp        0
ppltrst       0
lawobey       0
lrscale       0
stfgov        0
trstep        0
trstlgl       0
trstplc       0
trstplt       0
trstprl       0
trstprt    2919
trstun        0
trstsci       0
happy         0
dtype: int64


In [9]:
#Rellenamos la variable 'trstprt' con la moda
moda_trstprt = df_deuv2['trstprt'].mode()[0]
df_deuv2['trstprt'].fillna(moda_trstprt, inplace=True)

C:\Users\Josue;\AppData\Local\Temp\ipykernel_7156\2301469211.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_deuv2['trstprt'].fillna(moda_trstprt, inplace=True)


In [10]:
#Eliminamos los valores nulos de las columnas, que hacen referencia a que el encuestado no ha querido responder. valores 77,88,99
import pandas as pd
import numpy as np

def delnulos(df):
    """
    Filtra el DataFrame df_deuv2 y sobreescribe el DataFrame original
    con los resultados filtrados.
    """
    columnas_a_limpiar = ['pplhlp', 'ppltrst', 'lrscale', 'stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstun', 'trstsci', 'happy']
    for columna in columnas_a_limpiar:
        if columna in df.columns:
            df[columna] = np.where(df[columna] > 70, np.nan, df[columna])
            df.dropna(subset=[columna], inplace=True)
    return df

df_deuv2 = delnulos(df_deuv2)
print(df_deuv2.isnull().sum())

cntry      0
pplhlp     0
ppltrst    0
lawobey    0
lrscale    0
stfgov     0
trstep     0
trstlgl    0
trstplc    0
trstplt    0
trstprl    0
trstprt    0
trstun     0
trstsci    0
happy      0
dtype: int64


In [11]:
import pandas as pd
import numpy as np, random
#Reducimos el dataframe al 10%
tamano_muestra = int(len(df_deuv2) * 0.1)
np.random.seed(42)  
indices_aleatorios = np.random.choice(df_deuv2.index, size=tamano_muestra, replace=False)
df_deuv2_reducido = df_deuv2.loc[indices_aleatorios]
df_deuv2_reindexado = df_deuv2_reducido.reset_index(drop=True)
print(len(df_deuv2_reducido))

3153


In [12]:
#Borramos datos de la cache excepto nuestro dataframe final de Alemania.
del columnas_a_eliminar, datadeu, df_deuv2, df_deu, df_deuv2_reducido, nombres_columnas, indices_aleatorios

In [14]:
#Guardamos el dataframe de alemania en un csv en la carpeta processed
import pandas as pd
# Ruta completa al archivo CSV
ruta_completa_archivo = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/df_deu.csv'

# Guarda el DataFrame en el archivo CSV
df_deuv2_reindexado.to_csv(ruta_completa_archivo, index=False)

In [15]:
#Grupo 2 España (Media UE).
df_spa= pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/SpaData.csv')

In [16]:
#Eliminamos columnas del df que no interesen
columnas_a_eliminar = ['name', 'essround', 'edition', 'proddate', 'idno', 'dweight', 'pspwght', 'pweight', 'anweight', 'prob', 'stratum', 'psu']
df_spa.drop(columnas_a_eliminar, axis=1, inplace=True)

In [17]:
#Obtenemos una lista con los nombres de las columnas restantes.
nombres_columnas = df_spa.columns.tolist()
print(nombres_columnas)

['cntry', 'pplhlp', 'ppltrst', 'lawobey', 'lrscale', 'stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstun', 'trstsci', 'happy']


In [18]:
df_spa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21296 entries, 0 to 21295
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cntry    21296 non-null  object 
 1   pplhlp   21296 non-null  int64  
 2   ppltrst  21296 non-null  int64  
 3   lawobey  1729 non-null   float64
 4   lrscale  21296 non-null  int64  
 5   stfgov   21296 non-null  int64  
 6   trstep   21296 non-null  int64  
 7   trstlgl  21296 non-null  int64  
 8   trstplc  21296 non-null  int64  
 9   trstplt  21296 non-null  int64  
 10  trstprl  21296 non-null  int64  
 11  trstprt  19567 non-null  float64
 12  trstun   21296 non-null  int64  
 13  trstsci  2283 non-null   float64
 14  happy    21296 non-null  int64  
dtypes: float64(3), int64(11), object(1)
memory usage: 2.4+ MB


In [21]:
dataspa = {
    'lawobey': {
        1: 0.0222,
        2: 0.435,
        3: 0.23,
        4: 0.094,
        5: 0.02,
    },
    'trstsci': {
        0: 0.011,
        1: 0.003,
        2: 0.018,
        3: 0.028,
        4: 0.039,
        5: 0.075,
        6: 0.132,
        7: 0.217,
        8: 0.252,
        9: 0.136,
        10: 0.088,
    }
}

def rngpond(df, dataspa):
    """
    Rellena los datos faltantes en un DataFrame usando valores aleatorios
    generados según los pesos ponderados proporcionados en el diccionario datadeu.
    """
    for columna, pesos in dataspa.items():
        if columna in df.columns and df[columna].isnull().any():
            valores = np.array(list(pesos.keys()))
            pesos_ponderados = np.array(list(pesos.values()))

            # Normalizar los pesos ponderados
            suma_pesos = np.sum(pesos_ponderados)
            pesos_normalizados = pesos_ponderados / suma_pesos

            for index, row in df[df[columna].isnull()].iterrows():
                # Generar un valor aleatorio según los pesos ponderados normalizados
                valor_aleatorio = np.random.choice(valores, p=pesos_normalizados)

                # Rellenar el valor faltante con el valor aleatorio generado
                df.loc[index, columna] = valor_aleatorio
    return df



df_spav2 = rngpond(df_spa.copy(), dataspa)

print(df_spav2.isnull().sum())

cntry         0
pplhlp        0
ppltrst       0
lawobey       0
lrscale       0
stfgov        0
trstep        0
trstlgl       0
trstplc       0
trstplt       0
trstprl       0
trstprt    1729
trstun        0
trstsci       0
happy         0
dtype: int64


In [22]:
#Rellenamos la variable 'trstprt' con la moda
moda_trstprt = df_spav2['trstprt'].mode()[0]
df_spav2['trstprt'].fillna(moda_trstprt, inplace=True)

C:\Users\Josue;\AppData\Local\Temp\ipykernel_7156\349227089.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_spav2['trstprt'].fillna(moda_trstprt, inplace=True)


In [23]:
#Eliminamos los valores nulos de las columnas, que hacen referencia a que el encuestado no ha querido responder. valores 77,88,99
import pandas as pd
import numpy as np

def delnulos(df):
    """
    Filtra el DataFrame df_deuv2 y sobreescribe el DataFrame original
    con los resultados filtrados.
    """
    columnas_a_limpiar = ['pplhlp', 'ppltrst', 'lrscale', 'stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstun', 'trstsci', 'happy']
    for columna in columnas_a_limpiar:
        if columna in df.columns:
            df[columna] = np.where(df[columna] > 70, np.nan, df[columna])
            df.dropna(subset=[columna], inplace=True)
    return df

df_spav2= delnulos(df_spav2)
print(df_spav2.isnull().sum())

cntry      0
pplhlp     0
ppltrst    0
lawobey    0
lrscale    0
stfgov     0
trstep     0
trstlgl    0
trstplc    0
trstplt    0
trstprl    0
trstprt    0
trstun     0
trstsci    0
happy      0
dtype: int64


In [24]:
import pandas as pd
import numpy as np, random
#Reducimos el dataframe al 20%
tamano_muestra = int(len(df_spav2) * 0.2)
np.random.seed(42)  
indices_aleatorios = np.random.choice(df_spav2.index, size=tamano_muestra, replace=False)
df_spav2_reducido = df_spav2.loc[indices_aleatorios]
df_spav2_reindexado = df_spav2_reducido.reset_index(drop=True)
print(len(df_spav2_reindexado))

3219


In [ ]:
#Borramos datos de la cache excepto nuestro dataframe final de España.
del columnas_a_eliminar, dataspa, df_spav2, df_spa, df_spav2_reducido, nombres_columnas, indices_aleatorios

In [32]:
#Guardamos el dataframe de españa en un csv en la carpeta processed
import pandas as pd
ruta_completa_archivo = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/df_spa.csv'
df_spav2_reindexado.to_csv(ruta_completa_archivo, index=False)

In [27]:
#Grupo 3 Hungria(inferior  a la media UE):
df_hun= pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/HungData.csv')

In [28]:
#Eliminamos columnas del df que no interesen
columnas_a_eliminar = ['name', 'essround', 'edition', 'proddate', 'idno', 'dweight', 'pspwght', 'pweight', 'anweight', 'prob', 'stratum', 'psu']
df_hun.drop(columnas_a_eliminar, axis=1, inplace=True)

In [29]:
#Obtenemos una lista con los nombres de las columnas restantes.
nombres_columnas = df_hun.columns.tolist()
print(nombres_columnas)

['cntry', 'pplhlp', 'ppltrst', 'lawobey', 'lrscale', 'stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstun', 'trstsci', 'happy']


In [30]:
df_hun.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18760 entries, 0 to 18759
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cntry    18760 non-null  object 
 1   pplhlp   18760 non-null  int64  
 2   ppltrst  18760 non-null  int64  
 3   lawobey  1685 non-null   float64
 4   lrscale  18760 non-null  int64  
 5   stfgov   18760 non-null  int64  
 6   trstep   18760 non-null  int64  
 7   trstlgl  18760 non-null  int64  
 8   trstplc  18760 non-null  int64  
 9   trstplt  18760 non-null  int64  
 10  trstprl  18760 non-null  int64  
 11  trstprt  17075 non-null  float64
 12  trstun   18760 non-null  int64  
 13  trstsci  1849 non-null   float64
 14  happy    18760 non-null  int64  
dtypes: float64(3), int64(11), object(1)
memory usage: 2.1+ MB


In [31]:
datahun = {
    'lawobey': {
        1: 0.60,
        2: 0.326,
        3: 0.062,
        4: 0.011,
        5: 0.002,
    },
    'trstsci': {
        0: 0.011,
        1: 0.01,
        2: 0.029,
        3: 0.03,
        4: 0.053,
        5: 0.176,
        6: 0.121,
        7: 0.177,
        8: 0.197,
        9: 0.11,
        10: 0.086,
    }
}

def rngpond(df, datahun):
    """
    Rellena los datos faltantes en un DataFrame usando valores aleatorios
    generados según los pesos ponderados proporcionados en el diccionario datadeu.
    """
    for columna, pesos in datahun.items():
        if columna in df.columns and df[columna].isnull().any():
            valores = np.array(list(pesos.keys()))
            pesos_ponderados = np.array(list(pesos.values()))

            # Normalizar los pesos ponderados
            suma_pesos = np.sum(pesos_ponderados)
            pesos_normalizados = pesos_ponderados / suma_pesos

            for index, row in df[df[columna].isnull()].iterrows():
                # Generar un valor aleatorio según los pesos ponderados normalizados
                valor_aleatorio = np.random.choice(valores, p=pesos_normalizados)

                # Rellenar el valor faltante con el valor aleatorio generado
                df.loc[index, columna] = valor_aleatorio
    return df



df_hunv2 = rngpond(df_hun.copy(), datahun)

print(df_hunv2.isnull().sum())

cntry         0
pplhlp        0
ppltrst       0
lawobey       0
lrscale       0
stfgov        0
trstep        0
trstlgl       0
trstplc       0
trstplt       0
trstprl       0
trstprt    1685
trstun        0
trstsci       0
happy         0
dtype: int64
